# Preprocessing — Hotel Booking Cancellation

Preprocesado y feature engineering sobre `train.csv`/`test.csv`. Todos los parámetros de imputación, encoding y escalado se ajustan (fit) exclusivamente sobre train y se aplican (transform) sobre test.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train_df = pd.read_csv('../data_sample/train.csv')
test_df = pd.read_csv('../data_sample/test.csv')
train_df.shape, test_df.shape

((95512, 32), (23878, 32))

## Paso 1: Eliminar columnas con data leakage

`reservation_status` y `reservation_status_date` determinan `is_canceled` de forma directa (detectado en EDA) — se eliminan sí o sí antes de cualquier entrenamiento.

In [2]:
LEAKAGE_COLS = ['reservation_status', 'reservation_status_date']
train_df = train_df.drop(columns=LEAKAGE_COLS)
test_df = test_df.drop(columns=LEAKAGE_COLS)
assert not any(c in train_df.columns for c in LEAKAGE_COLS)
print('columnas leakage eliminadas de train y test')

columnas leakage eliminadas de train y test


## Paso 2: Duplicados

Se eliminan duplicados exactos en train (calidad de datos). Test se deja intacto para no alterar el conjunto de evaluación.

In [3]:
dups = train_df.duplicated().sum()
print(f'duplicados en train: {dups}')
train_df = train_df.drop_duplicates().reset_index(drop=True)
train_df.shape

duplicados en train: 24463


(71049, 30)

## Paso 3: Feature engineering

- `has_agent`: indicador binario de si la reserva vino con agente (el missing en `agent` es informativo, visto en EDA).
- `has_company`: indicador binario equivalente para `company` (94.3% missing, se elimina la columna original por escaso valor más allá del indicador).
- `total_nights`: noches totales (weekend + week).
- `total_guests`: adultos + niños + bebés.

In [4]:
def feature_engineer(df):
    df = df.copy()
    df['has_agent'] = df['agent'].notna().astype(int)
    df['has_company'] = df['company'].notna().astype(int)
    df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
    df['total_guests'] = df['adults'] + df['children'].fillna(0) + df['babies']
    df = df.drop(columns=['agent', 'company'])
    return df

train_df = feature_engineer(train_df)
test_df = feature_engineer(test_df)
train_df[['has_agent','has_company','total_nights','total_guests']].describe()

,has_agent,has_company,total_nights,total_guests
count,71049.000000,71049.000000,71049.000000,71049.000000
mean,0.861588,0.059635,3.621726,2.020338
std,0.345334,0.236811,2.756847,0.795137
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,2.000000,2.000000
50%,1.000000,0.000000,3.000000,2.000000
75%,1.000000,0.000000,5.000000,2.000000
max,1.000000,1.000000,69.000000,55.000000


## Paso 4: Tratamiento de outliers en `adr`

EDA detectó un valor negativo (-6.38) y un extremo muy alto (5400) en `adr`. Se capea (winsoriza) usando percentiles 1 y 99 calculados **solo sobre train**, y se aplican los mismos límites a test.

In [5]:
lower = train_df['adr'].quantile(0.01)
upper = train_df['adr'].quantile(0.99)
print(f'limites adr (train): [{lower:.2f}, {upper:.2f}]')

train_df['adr'] = train_df['adr'].clip(lower, upper)
test_df['adr'] = test_df['adr'].clip(lower, upper)
train_df['adr'].describe()

limites adr (train): [0.00, 260.26]


count    71049.000000
mean       105.628216
std         50.519467
min          0.000000
25%         72.000000
50%         98.000000
75%        133.300000
max        260.260000
Name: adr, dtype: float64

## Paso 5: Agrupar categorías de alta cardinalidad en `country`

`country` tiene demasiadas categorías para one-hot directo. Se conservan las 15 más frecuentes (calculadas sobre train) y el resto se agrupa como `Other`.

In [6]:
top_countries = train_df['country'].value_counts().head(15).index.tolist()
print(top_countries)

def group_country(df):
    df = df.copy()
    df['country'] = df['country'].where(df['country'].isin(top_countries), 'Other')
    return df

train_df = group_country(train_df)
test_df = group_country(test_df)
train_df['country'].value_counts()

['PRT', 'GBR', 'FRA', 'ESP', 'DEU', 'ITA', 'IRL', 'BEL', 'BRA', 'NLD', 'USA', 'CHE', 'CN', 'AUT', 'SWE']


country
PRT      22341
GBR       8463
Other     7873
FRA       7200
ESP       5874
DEU       4409
ITA       2476
IRL       2424
BEL       1700
BRA       1615
NLD       1558
USA       1506
CHE       1272
CN         879
AUT        776
SWE        683
Name: count, dtype: int64

## Paso 6: Separar X/y y definir columnas por tipo

In [7]:
TARGET = 'is_canceled'
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

cat_cols = X_train.select_dtypes(include=['object','str']).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
print('categoricas:', cat_cols)
print('numericas:', num_cols)

categoricas: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']
numericas: ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'has_agent', 'has_company', 'total_nights', 'total_guests']


## Paso 7: Pipeline de imputación + encoding + escalado

- Numéricas: imputación por mediana + `StandardScaler`.
- Categóricas: imputación por moda + `OneHotEncoder` (nominal, sin orden).
- Todo el `fit` se hace sobre train; test solo se transforma.

In [8]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols),
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
X_train_proc.shape, X_test_proc.shape

((71049, 98), (23878, 98))

In [9]:
feature_names = preprocessor.get_feature_names_out()
X_train_final = pd.DataFrame(X_train_proc, columns=feature_names)
X_test_final = pd.DataFrame(X_test_proc, columns=feature_names)
X_train_final.shape, X_test_final.shape

((71049, 98), (23878, 98))

## Paso 8: Guardar datasets procesados y transformador

In [10]:
X_train_final.to_csv('../data_sample/X_train.csv', index=False)
X_test_final.to_csv('../data_sample/X_test.csv', index=False)
y_train.to_csv('../data_sample/y_train.csv', index=False)
y_test.to_csv('../data_sample/y_test.csv', index=False)

joblib.dump(preprocessor, '../models/preprocessor.joblib')
print('guardados: X_train.csv, X_test.csv, y_train.csv, y_test.csv, models/preprocessor.joblib')

guardados: X_train.csv, X_test.csv, y_train.csv, y_test.csv, models/preprocessor.joblib


## Resumen para modelado

- `X_train.csv` / `X_test.csv`: features numéricas escaladas + categóricas one-hot, listas para entrenar (sin fugas — encoder/scaler ajustados solo en train).
- `y_train.csv` / `y_test.csv`: target `is_canceled`.
- `src/models/preprocessor.joblib`: `ColumnTransformer` ajustado, reutilizable para inferencia sobre datos nuevos.
- Eliminadas: `reservation_status`, `reservation_status_date` (leakage), `agent`, `company` (sustituidas por indicadores `has_agent`/`has_company`).
- Nuevas features: `has_agent`, `has_company`, `total_nights`, `total_guests`.
- `adr` capeado a percentiles [1, 99] de train.